# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Wed Sep 09 19:16:52 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671184,35.9,1479548,79.1,1479548,79.1
Vcells,1242574,9.5,8388608,64.0,1978697,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [4]:
# Configuración general de la corrida
PARAM <- list()

# Cambiar estos valores para cada escenario
PARAM$experimento <- 9140
PARAM$nombre_escenario <- "M2_sin_febrero_marzo_2020"
PARAM$meses_excluidos <- c(
  202002,
  202003
)

# Semillas definidas explícitamente
PARAM$semillas <- c(
  700001L,
  690506L,
  804528L,
  613037L,
  648983L,
  365305L,
  163687L
)

PARAM$cantidad_semillas <- length(PARAM$semillas)
PARAM$semilla_primigenia <- PARAM$semillas[1]

PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

# TRUE: genera archivos y los sube.
# FALSE: solamente genera los archivos.
PARAM$subir_kaggle <- TRUE

# Validaciones
stopifnot(PARAM$cantidad_semillas >= 1L)
stopifnot(!anyDuplicated(PARAM$semillas))
stopifnot(all(PARAM$semillas > 0L))

# Muestra las semillas que utilizará
PARAM$semillas


[1] 700001 690506 804528 613037 648983 365305 163687

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [27]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [28]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [29]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "kmes"                              
[57] "mpayroll_sobre_edad"

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [30]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [31]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [32]:
ncol(dataset)
colnames(dataset)

[1] 273

[1] "numero_de_cliente"                        
  [2] "foto_mes"                                 
  [3] "internet"                                 
  [4] "cliente_edad"                             
  [5] "cliente_antiguedad"                       
  [6] "mrentabilidad"                            
  [7] "mrentabilidad_annual"                     
  [8] "mcomisiones"                              
  [9] "mactivos_margen"                          
 [10] "mpasivos_margen"                          
 [11] "cproductos"                               
 [12] "mcuenta_corriente"                        
 [13] "mcaja_ahorro"                             
 [14] "cdescubierto_preacordado"                 
 [15] "mcuentas_saldo"                           
 [16] "ctarjeta_visa"                            
 [17] "ctarjeta_visa_transacciones"              
 [18] "mtarjeta_visa_consumo"                    
 [19] "ctarjeta_master"                          
 [20] "ctarjeta_master_transacciones"            
 [21] "mtarjeta_master_consumo"                  
 [22] "cprestamos_personales"                    
 [23] "mprestamos_personales"                    
 [24] "cpayroll_trx"                             
 [25] "mpayroll"                                 
 [26] "mttarjeta_visa_debitos_automaticos"       
 [27] "ccomisiones_mantenimiento"                
 [28] "mcomisiones_mantenimiento"                
 [29] "ccomisiones_otras"                        
 [30] "mtransferencias_recibidas"                
 [31] "ccallcenter_transacciones"                
 [32] "thomebanking"                             
 [33] "chomebanking_transacciones"               
 [34] "ctrx_quarter"                             
 [35] "Master_status"                            
 [36] "Master_mfinanciacion_limite"              
 [37] "Master_Fvencimiento"                      
 [38] "Master_msaldototal"                       
 [39] "Master_mlimitecompra"                     
 [40] "Master_fultimo_cierre"                    
 [41] "Master_fechaalta"                         
 [42] "Master_mconsumototal"                     
 [43] "Master_cconsumos"                         
 [44] "Master_mpagominimo"                       
 [45] "Visa_status"                              
 [46] "Visa_mfinanciacion_limite"                
 [47] "Visa_Fvencimiento"                        
 [48] "Visa_msaldototal"                         
 [49] "Visa_mlimitecompra"                       
 [50] "Visa_fultimo_cierre"                      
 [51] "Visa_fechaalta"                           
 [52] "Visa_mconsumototal"                       
 [53] "Visa_cconsumos"                           
 [54] "Visa_mpagominimo"                         
 [55] "clase_ternaria"                           
 [56] "kmes"                                     
 [57] "mpayroll_sobre_edad"                      
 [58] "internet_lag1"                            
 [59] "cliente_edad_lag1"                        
 [60] "cliente_antiguedad_lag1"                  
 [61] "mrentabilidad_lag1"                       
 [62] "mrentabilidad_annual_lag1"                
 [63] "mcomisiones_lag1"                         
 [64] "mactivos_margen_lag1"                     
 [65] "mpasivos_margen_lag1"                     
 [66] "cproductos_lag1"                          
 [67] "mcuenta_corriente_lag1"                   
 [68] "mcaja_ahorro_lag1"                        
 [69] "cdescubierto_preacordado_lag1"            
 [70] "mcuentas_saldo_lag1"                      
 [71] "ctarjeta_visa_lag1"                       
 [72] "ctarjeta_visa_transacciones_lag1"         
 [73] "mtarjeta_visa_consumo_lag1"               
 [74] "ctarjeta_master_lag1"                     
 [75] "ctarjeta_master_transacciones_lag1"       
 [76] "mtarjeta_master_consumo_lag1"             
 [77] "cprestamos_personales_lag1"               
 [78] "mprestamos_personales_lag1"               
 [79] "cpayroll_trx_lag1"                        
 [80] "mpayroll_lag1"                            
 [

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [33]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se evalúa una estrategia temporal configurable. Los meses indicados en
`PARAM$meses_excluidos` se eliminan tanto del entrenamiento de Modelado como del
entrenamiento final de Producción, según la consigna del Experimento 4.


In [34]:
PARAM$trainingstrategy$validate <- c(202107)

meses_training_base <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)

PARAM$trainingstrategy$training <- setdiff(
  meses_training_base,
  PARAM$meses_excluidos
)

PARAM$trainingstrategy$training_pct <- 1.0
PARAM$trainingstrategy$positivos <- c("BAJA+1", "BAJA+2")

cat("Escenario:", PARAM$nombre_escenario, "\n")
cat("Meses excluidos:", PARAM$meses_excluidos, "\n")
cat("Meses de training:", PARAM$trainingstrategy$training, "\n")


Escenario: M2_sin_febrero_marzo_2020 
Meses excluidos: 202002 202003 
Meses de training: 201901 201902 201903 201904 201905 201906 201907 201908 201909 201910 201911 201912 202001 202004 202005 202006 202007 202008 202009 202010 202011 202012 202101 202102 202103 202104 202105 


In [35]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [36]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [37]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

Loading required package: lightgbm



In [38]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 32938

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parámetros optimizados mediante Grid Search multisemilla:
  * `num_leaves`: 64, 128, 256, 384 y 512
  * `min_data_in_leaf`: 64, 256, 512, 1024 y 2048
  * `feature_fraction`: 0.5 y 0.8

Cada una de las 50 configuraciones se evalúa con todas las semillas configuradas.


In [39]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [40]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [41]:
# Producto cartesiano: 50 configuraciones por cada semilla
tb_nueva <- CJ(
  num_leaves = c(64, 128, 256, 384, 512),
  min_data_in_leaf = c(64, 256, 512, 1024, 2048),
  feature_fraction = c(0.5, 0.8),
  seed = PARAM$semillas
)

tb_nueva[, `:=`(AUC = NA_real_, num_iterations = NA_integer_)]

PARAM$archivo_grid_detalle <- paste0(
  "grid_detalle_", PARAM$nombre_escenario, ".txt"
)

# Si existe un checkpoint compatible, recupera lo ya calculado.
if (file.exists(PARAM$archivo_grid_detalle)) {
  checkpoint <- fread(PARAM$archivo_grid_detalle)
  claves <- c("num_leaves", "min_data_in_leaf", "feature_fraction", "seed")

  if (nrow(checkpoint) == nrow(tb_nueva) &&
      fsetequal(checkpoint[, ..claves], tb_nueva[, ..claves])) {
    setkeyv(tb_nueva, claves)
    setkeyv(checkpoint, claves)
    tb_nueva[checkpoint, `:=`(
      AUC = i.AUC,
      num_iterations = i.num_iterations
    )]
    setorderv(tb_nueva, claves)
    cat("Checkpoint recuperado. Resultados completos:",
        sum(!is.na(tb_nueva$AUC)), "\n")
  } else {
    stop("El checkpoint no coincide con las semillas o la grilla actual")
  }
}

nrow(tb_nueva)


[1] 350

##### Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 65 minutos
<br> una Analista Jr  debe ser capaz de tolerar estoicamente esta tortura
<br> (y masticar chicle al mismo tiempo)

In [42]:
# Calcula solamente combinaciones pendientes y guarda después de cada modelo.
pendientes <- which(is.na(tb_nueva$AUC))

for (i in pendientes) {
  x <- as.list(tb_nueva[i, .(
    num_leaves,
    min_data_in_leaf,
    feature_fraction,
    seed
  )])

  resultado <- Estimar_AUC_lightgbm(x)
  set(tb_nueva, i = i, j = "AUC", value = as.numeric(resultado[[1]]))
  set(tb_nueva, i = i, j = "num_iterations",
      value = as.integer(resultado[[2]]))

  fwrite(tb_nueva, PARAM$archivo_grid_detalle, sep = "\t")
}

stopifnot(sum(is.na(tb_nueva$AUC)) == 0L)
cat("Grid Search completo:", nrow(tb_nueva), "modelos\n")


Wed Sep 09 19:17:49 2026  64, 64, 0.5, 163687 niter 95 AUC 0.931229130253651

Wed Sep 09 19:18:29 2026  64, 64, 0.5, 365305 niter 78 AUC 0.93004174213611

Wed Sep 09 19:19:06 2026  64, 64, 0.5, 613037 niter 94 AUC 0.930550037430729

Wed Sep 09 19:19:45 2026  64, 64, 0.5, 648983 niter 113 AUC 0.931189181673323

Wed Sep 09 19:20:21 2026  64, 64, 0.5, 690506 niter 92 AUC 0.932153366598044

Wed Sep 09 19:21:03 2026  64, 64, 0.5, 700001 niter 102 AUC 0.929946347966211

Wed Sep 09 19:21:40 2026  64, 64, 0.5, 804528 niter 102 AUC 0.929350159186336

Wed Sep 09 19:22:05 2026  64, 64, 0.8, 163687 niter 171 AUC 0.930197373081807

Wed Sep 09 19:22:31 2026  64, 64, 0.8, 365305 niter 86 AUC 0.929576435346874

Wed Sep 09 19:22:58 2026  64, 64, 0.8, 613037 niter 209 AUC 0.929870845479816

Wed Sep 09 19:23:18 2026  64, 64, 0.8, 648983 niter 87 AUC 0.930484216444779

Wed Sep 09 19:23:45 2026  64, 64, 0.8, 690506 niter 200 AUC 0.930282953580609

Wed Sep 09 19:24:05 2026  64, 64, 0.8, 700001 niter 86 AUC 

Grid Search completo: 350 modelos


la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [43]:
# Guarda el detalle completo sin duplicar ejecuciones anteriores.
fwrite(
  tb_nueva,
  file = PARAM$archivo_grid_detalle,
  sep = "\t"
)

tb_nueva


num_leaves,min_data_in_leaf,feature_fraction,seed,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
64,64,0.5,163687,0.9312291,95
64,64,0.5,365305,0.9300417,78
64,64,0.5,613037,0.9305500,94
64,64,0.5,648983,0.9311892,113
64,64,0.5,690506,0.9321534,92
64,64,0.5,700001,0.9299463,102
64,64,0.5,804528,0.9293502,102
64,64,0.8,163687,0.9301974,171
64,64,0.8,365305,0.9295764,86


In [44]:
# Resume por configuración y selecciona por AUC medio entre semillas.
tb_resumen <- tb_nueva[, .(
  AUC_media = mean(AUC),
  AUC_mediana = median(AUC),
  AUC_sd = sd(AUC),
  AUC_min = min(AUC),
  AUC_max = max(AUC),
  iter_media = mean(num_iterations),
  iter_mediana = median(num_iterations)
), by = .(num_leaves, min_data_in_leaf, feature_fraction)]

setorder(tb_resumen, -AUC_media, AUC_sd, num_leaves)
mejor <- tb_resumen[1]

PARAM$out$lgbm$AUC <- mejor$AUC_media
PARAM$out$lgbm$mejores_hiperparametros <- list(
  num_leaves = mejor$num_leaves,
  min_data_in_leaf = mejor$min_data_in_leaf,
  feature_fraction = mejor$feature_fraction,
  num_iterations = as.integer(ceiling(mejor$iter_mediana))
)

PARAM$archivo_grid_resumen <- paste0(
  "grid_resumen_", PARAM$nombre_escenario, ".txt"
)
fwrite(tb_resumen, PARAM$archivo_grid_resumen, sep = "\t")

cat("Mejor configuración del escenario:\n")
print(PARAM$out$lgbm$mejores_hiperparametros)
tb_resumen[1:10]


Mejor configuración del escenario:
$num_leaves
[1] 384

$min_data_in_leaf
[1] 1024

$feature_fraction
[1] 0.8

$num_iterations
[1] 102



num_leaves,min_data_in_leaf,feature_fraction,AUC_media,AUC_mediana,AUC_sd,AUC_min,AUC_max,iter_media,iter_mediana
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
384,1024,0.8,0.9336215,0.9333329,0.0010410829,0.9324040,0.9350672,107.14286,102
512,1024,0.8,0.9335903,0.9335620,0.0010497911,0.9323009,0.9352362,107.57143,107
512,1024,0.5,0.9334079,0.9334508,0.0009460471,0.9322004,0.9351379,107.00000,106
256,2048,0.8,0.9330863,0.9331726,0.0005061689,0.9323065,0.9338744,131.00000,133
384,1024,0.5,0.9330786,0.9331450,0.0006663141,0.9321050,0.9338405,97.00000,97
256,1024,0.5,0.9329941,0.9330848,0.0007801456,0.9320222,0.9342044,99.71429,103
128,1024,0.8,0.9329543,0.9328379,0.0005105699,0.9323954,0.9338337,111.14286,114
384,2048,0.8,0.9329192,0.9328359,0.0005582817,0.9322512,0.9339849,131.14286,133
512,2048,0.8,0.9329192,0.9328359,0.0005582817,0.9322512,0.9339849,131.14286,133


In [45]:
tb_nueva

num_leaves,min_data_in_leaf,feature_fraction,seed,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
64,64,0.5,163687,0.9312291,95
64,64,0.5,365305,0.9300417,78
64,64,0.5,613037,0.9305500,94
64,64,0.5,648983,0.9311892,113
64,64,0.5,690506,0.9321534,92
64,64,0.5,700001,0.9299463,102
64,64,0.5,804528,0.9293502,102
64,64,0.8,163687,0.9301974,171
64,64,0.8,365305,0.9295764,86


### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización

In [46]:
meses_final_train_base <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)

PARAM$trainingstrategy$final_train <- setdiff(
  meses_final_train_base,
  PARAM$meses_excluidos
)

dataset[, fold_final_train :=
  foto_mes %in% PARAM$trainingstrategy$final_train]

dfinal_train <- lgb.Dataset(
  data = data.matrix(
    dataset[fold_final_train == TRUE, campos_buenos, with = FALSE]
  ),
  label = dataset[fold_final_train == TRUE, clase01],
  free_raw_data = TRUE
)

cat("Filas de final training:",
    nrow(dataset[fold_final_train == TRUE]), "\n")


Filas de final training: 851888 


##### Final Training Hyperparameters

In [47]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [48]:
# Producción completa para todas las semillas configuradas.
if (!require("yaml")) install.packages("yaml")
require("yaml")

PARAM$trainingstrategy$future <- c(202109)
PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)

dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]

if (PARAM$subir_kaggle &&
    length(PARAM$semillas) * length(PARAM$kaggle$cortes) > 100L) {
  warning("La corrida requiere más de 100 submits; Kaggle limita los envíos diarios")
}

tb_produccion <- data.table()

for (semilla in PARAM$semillas) {
  cat("\nProducción - escenario", PARAM$nombre_escenario,
      "- semilla", semilla, "\n")

  carpeta_semilla <- file.path(paste0("semilla_", semilla))
  carpeta_kaggle <- file.path(carpeta_semilla, "kaggle")
  dir.create(carpeta_kaggle, recursive = TRUE, showWarnings = FALSE)

  param_semilla <- copy(param_final)
  param_semilla$seed <- semilla

  final_model <- lgb.train(
    data = dfinal_train,
    param = param_semilla,
    verbose = -100
  )

  lgb.save(final_model, file.path(carpeta_semilla, "modelo.txt"))

  tb_importancia <- as.data.table(lgb.importance(final_model))
  fwrite(
    tb_importancia,
    file.path(carpeta_semilla, "importancia.txt"),
    sep = "\t"
  )

  prediccion <- predict(
    final_model,
    data.matrix(dfuture[, campos_buenos, with = FALSE])
  )

  tb_prediccion <- dfuture[, .(numero_de_cliente)]
  tb_prediccion[, prob := prediccion]
  setorder(tb_prediccion, -prob)

  fwrite(
    tb_prediccion,
    file.path(carpeta_semilla, "prediccion.txt"),
    sep = "\t"
  )

  for (envios in PARAM$kaggle$cortes) {
    tb_prediccion[, Predicted := 0L]
    tb_prediccion[seq_len(envios), Predicted := 1L]

    nombre_archivo <- paste0(
      "KA", PARAM$experimento, "_",
      PARAM$nombre_escenario, "_S", semilla, "_", envios, ".csv"
    )
    archivo_kaggle <- file.path(carpeta_kaggle, nombre_archivo)

    fwrite(
      tb_prediccion[, .(numero_de_cliente, Predicted)],
      archivo_kaggle,
      sep = ","
    )

    estado <- "archivo_generado"
    respuesta <- ""

    if (PARAM$subir_kaggle) {
      mensaje <- paste0(
        "exp=", PARAM$experimento,
        " escenario=", PARAM$nombre_escenario,
        " semilla=", semilla,
        " envios=", envios
      )

      respuesta <- tryCatch(
        system2(
          "kaggle",
          args = c(
            "competitions", "submit",
            "-c", PARAM$kaggle$competencia,
            "-f", shQuote(archivo_kaggle),
            "-m", shQuote(mensaje)
          ),
          stdout = TRUE,
          stderr = TRUE
        ),
        error = function(e) paste("ERROR:", conditionMessage(e))
      )

      estado <- paste(respuesta, collapse = " | ")
      cat(estado, "\n")
      Sys.sleep(30)
    }

    tb_produccion <- rbind(
      tb_produccion,
      data.table(
        experimento = PARAM$experimento,
        escenario = PARAM$nombre_escenario,
        semilla = semilla,
        envios = envios,
        archivo = archivo_kaggle,
        estado = estado
      )
    )

    fwrite(
      tb_produccion,
      paste0("produccion_", PARAM$nombre_escenario, ".txt"),
      sep = "\t"
    )
  }

  rm(final_model, prediccion, tb_prediccion, tb_importancia)
  gc(full = TRUE, verbose = FALSE)
}

write_yaml(
  PARAM,
  file = paste0("PARAM_", PARAM$nombre_escenario, ".yml")
)

cat("Producción finalizada para", length(PARAM$semillas), "semillas\n")
tb_produccion


Loading required package: yaml




Producción - escenario M2_sin_febrero_marzo_2020 - semilla 700001 
100%|██████████| 355k/355k [00:00<00:00, 779kB/s] | 0 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 


Warning message in system2("kaggle", args = c("competitions", "submit", "-c", PARAM$kaggle$competencia, :
“running command ''kaggle' competitions submit -c utn-2026-virtual-jr -f 'semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_1900.csv' -m 'exp=9140 escenario=M2_sin_febrero_marzo_2020 semilla=700001 envios=1900' 2>&1' had status 1”


100%|██████████| 355k/355k [00:00<00:00, 812kB/s] | 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission 


Warning message in system2("kaggle", args = c("competitions", "submit", "-c", PARAM$kaggle$competencia, :
“running command ''kaggle' competitions submit -c utn-2026-virtual-jr -f 'semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_2000.csv' -m 'exp=9140 escenario=M2_sin_febrero_marzo_2020 semilla=700001 envios=2000' 2>&1' had status 1”


...

Warning message in system2("kaggle", args = c("competitions", "submit", "-c", PARAM$kaggle$competencia, :
“running command ''kaggle' competitions submit -c utn-2026-virtual-jr -f 'semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_2100.csv' -m 'exp=9140 escenario=M2_sin_febrero_marzo_2020 semilla=700001 envios=2100' 2>&1' had status 1”


100%|██████████| 355k/355k [00:00<00:00, 840kB/s] | 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission 


Warning message in system2("kaggle", args = c("competitions", "submit", "-c", PARAM$kaggle$competencia, :
“running command ''kaggle' competitions submit -c utn-2026-virtual-jr -f 'semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_2200.csv' -m 'exp=9140 escenario=M2_sin_febrero_marzo_2020 semilla=700001 envios=2200' 2>&1' had status 1”


100%|██████████| 355k/355k [00:00<00:00, 815kB/s] | 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission 
100%|██████████| 355k/355k [00:00<00:00, 814kB/s] | 99 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 
100%|██████████| 355k/355k [00:00<00:00, 812kB/s] | 98 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 

Producción - escenario M2_sin_febrero_marzo_2020 - semilla 690506 
100%|██████████| 355k/355k [00:00<00:00, 809kB/s] | 97 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 
100%|██████████| 355k/355k [00:00<00:00, 815kB/s] | 96 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 
100%|██████████| 355k/355k [00:00<00:00, 686kB/s] | 95 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 
100%|██████████| 355k/355k [00:00<00:00, 742kB/s]  | 94 submissions remaining today. | Successful

experimento,escenario,semilla,envios,archivo,estado
<dbl>,<chr>,<int>,<dbl>,<chr>,<chr>
9140,M2_sin_febrero_marzo_2020,700001,1800,semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_1800.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 779kB/s] | 0 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr"
9140,M2_sin_febrero_marzo_2020,700001,1900,semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_1900.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 812kB/s] | 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission"
9140,M2_sin_febrero_marzo_2020,700001,2000,semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_2000.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 792kB/s] | 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission"
9140,M2_sin_febrero_marzo_2020,700001,2100,semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_2100.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 840kB/s] | 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission"
9140,M2_sin_febrero_marzo_2020,700001,2200,semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_2200.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 815kB/s] | 400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission"
9140,M2_sin_febrero_marzo_2020,700001,2300,semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_2300.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 814kB/s] | 99 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr"
9140,M2_sin_febrero_marzo_2020,700001,2400,semilla_700001/kaggle/KA9140_M2_sin_febrero_marzo_2020_S700001_2400.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 812kB/s] | 98 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr"
9140,M2_sin_febrero_marzo_2020,690506,1800,semilla_690506/kaggle/KA9140_M2_sin_febrero_marzo_2020_S690506_1800.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 809kB/s] | 97 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr"
9140,M2_sin_febrero_marzo_2020,690506,1900,semilla_690506/kaggle/KA9140_M2_sin_febrero_marzo_2020_S690506_1900.csv,"0%| | 0.00/355k [00:00<?, ?B/s] 100%|██████████| 355k/355k [00:00<00:00, 815kB/s] | 96 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr"


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


*Esta etapa se ejecutó automáticamente en la celda de Producción multisemilla anterior.*


In [49]:
format(Sys.time(), "%a %b %d %X %Y")


[1] "Thu Sep 10 00:25:31 2026"